In [1]:
!pip install streamlit pyngrok sentence-transformers faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 76.7 MB/s eta 0:00:00


In [2]:
!ngrok authtoken 3EqQSYRF2zjLKguiHR5rDwQ6oYu_4phpB82AJvhfy7XbvLgBG

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [3]:
from pyngrok import ngrok
import subprocess
import time
ngrok.kill()
proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true", "--server.enableCORS", "false",],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(5)
public_url=ngrok.connect(8501)
print(f"\n{'='*50}")
print(f"App is live ast: {public_url}")
print(f"{'='*50}")


App is live ast: NgrokTunnel: "https://goal-gondola-geologic.ngrok-free.dev" -> "http://localhost:8501"


In [4]:
import shutil
import os
ARTIFACTS = {
    "/kaggle/input/notebooks/anymansince2005/baseline-model/tfidf_matrix.npz"           : "/kaggle/working/tfidf_matrix.npz",
    "/kaggle/input/notebooks/anymansince2005/baseline-model/tfidf_vectorizer.pkl"       : "/kaggle/working/tfidf_vectorizer.pkl",

    "/kaggle/input/notebooks/anymansince2005/semantic-embedding/embeddings.npy"             : "/kaggle/working/embeddings.npy",
    "/kaggle/input/notebooks/anymansince2005/semantic-embedding/faiss_index.bin"            : "/kaggle/working/faiss_index.bin",

    "/kaggle/input/notebooks/anymansince2005/citation-graphs/citation_graph.pkl"         : "/kaggle/working/citation_graph.pkl",
    "/kaggle/input/notebooks/anymansince2005/citation-graphs/arxiv_ml_graph_features.parquet": "/kaggle/working/arxiv_ml_final.parquet",

    "/kaggle/input/notebooks/anymansince2005/user-modeling/user_profiles.pkl"          : "/kaggle/working/user_profiles.pkl",

    "/kaggle/input/notebooks/anymansince2005/mmr-reranking/pipeline_config.json"       : "/kaggle/working/pipeline_config.json",
    "/kaggle/input/notebooks/anymansince2005/mmr-reranking/all_metrics.json"           : "/kaggle/working/all_metrics.json",
    "/kaggle/input/notebooks/anymansince2005/mmr-reranking/sample_recommendations.parquet": "/kaggle/working/sample_recommendations.parquet",
}

for src, dst in ARTIFACTS.items():
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"✓ {os.path.basename(src)}")
    else:
        print(f"✗ NOT FOUND: {src}")

✓ tfidf_matrix.npz
✓ tfidf_vectorizer.pkl
✓ embeddings.npy
✓ faiss_index.bin
✓ citation_graph.pkl
✓ arxiv_ml_graph_features.parquet
✓ user_profiles.pkl
✓ pipeline_config.json
✓ all_metrics.json
✓ sample_recommendations.parquet


In [5]:
import os

REQUIRED = [
    "embeddings.npy",
    "faiss_index.bin",
    "citation_graph.pkl",
    "arxiv_ml_final.parquet",
    "pipeline_config.json",
    "all_metrics.json",
]

print("Artifact check:")
all_good = True
for f in REQUIRED:
    path   = f"/kaggle/working/{f}"
    exists = os.path.exists(path)
    size   = f"{os.path.getsize(path)/1e6:.1f}MB" if exists else "—"
    status = "✓" if exists else "✗ MISSING"
    print(f"  {status}  {f:<45} {size}")
    if not exists:
        all_good = False

print(f"\n{'All good — ready to launch' if all_good else 'Fix missing files before continuing'}")

Artifact check:
  ✓  embeddings.npy                                0.2MB
  ✓  faiss_index.bin                               0.2MB
  ✓  citation_graph.pkl                            0.0MB
  ✓  arxiv_ml_final.parquet                        0.1MB
  ✓  pipeline_config.json                          0.0MB
  ✓  all_metrics.json                              0.0MB

All good — ready to launch


In [6]:
%%writefile /kaggle/working/app.py

import streamlit as st
import pandas as pd
import numpy as np
import faiss
import pickle
import json
from sentence_transformers import SentenceTransformer
from collections import defaultdict

st.set_page_config(
    page_title = "Research Paper Recommender",
    page_icon  = "📄",
    layout     = "wide",
)

ARTIFACTS_DIR = "/kaggle/working/"

ML_CATEGORIES = {
    "cs.LG" : "Machine Learning",
    "cs.AI" : "Artificial Intelligence",
    "stat.ML": "Stats ML",
    "cs.CL" : "Computation & Language",
    "cs.CV" : "Computer Vision",
    "cs.NE" : "Neural & Evolutionary",
}

@st.cache_resource(show_spinner="Loading model...")
def load_model():
    return SentenceTransformer("all-MiniLM-L6-v2")

@st.cache_resource(show_spinner="Loading data...")
def load_artifacts():
    df          = pd.read_parquet(f"{ARTIFACTS_DIR}arxiv_ml_final.parquet")
    df          = df.reset_index(drop=True)
    embeddings  = np.load(f"{ARTIFACTS_DIR}embeddings.npy")
    faiss_index = faiss.read_index(f"{ARTIFACTS_DIR}faiss_index.bin")

    with open(f"{ARTIFACTS_DIR}citation_graph.pkl", "rb") as f:
        G = pickle.load(f)

    with open(f"{ARTIFACTS_DIR}pipeline_config.json") as f:
        config = json.load(f)

    return df, embeddings, faiss_index, G, config

model                                  = load_model()
df, embeddings, faiss_index, G, config = load_artifacts()

def maximal_marginal_relevance(query_vec, candidate_idxs, embeddings, top_n=10, lambda_param=0.60):
    query_vec  = query_vec.flatten()
    cand_vecs  = embeddings[candidate_idxs]
    rel_scores = cand_vecs @ query_vec

    selected, selected_vecs = [], []
    remaining = list(zip(candidate_idxs, rel_scores))

    while remaining and len(selected) < top_n:
        if not selected_vecs:
            best_idx = int(np.argmax([r for _, r in remaining]))
        else:
            sel_mat    = np.array(selected_vecs)
            mmr_scores = [
                lambda_param * rel - (1 - lambda_param) * float((sel_mat @ embeddings[ci]).max())
                for ci, rel in remaining
            ]
            best_idx = int(np.argmax(mmr_scores))

        best_paper_idx, best_rel = remaining.pop(best_idx)
        selected.append((best_paper_idx, round(float(best_rel), 4)))
        selected_vecs.append(embeddings[best_paper_idx])

    return selected


def apply_recency_boost(recs, half_life_years=3.0, boost_weight=0.15):
    recs            = recs.copy()
    now             = pd.Timestamp.now()
    recs["date"]    = pd.to_datetime(recs["date"], errors="coerce")
    age_years       = (now - recs["date"]).dt.days / 365.25
    recs["recency"] = np.exp(-np.log(2) * age_years / half_life_years).clip(0, 1).round(4)
    rel             = recs["relevance_score"]
    rel_norm        = (rel - rel.min()) / (rel.max() - rel.min() + 1e-9)
    recs["final_score"] = (
        (1 - boost_weight) * rel_norm + boost_weight * recs["recency"]
    ).round(4)
    return recs.sort_values("final_score", ascending=False).reset_index(drop=True)


def generate_explanation(query_idx, rec_idx, df, G, sem_score):
    rec_row      = df.iloc[rec_idx]
    if G.has_edge(query_idx, rec_idx) or G.has_edge(rec_idx, query_idx):
        return "🔗 Frequently cited with this paper"
    pr_threshold = df["graph_pagerank"].quantile(0.85)
    if sem_score > 0.80 and rec_row["graph_pagerank"] >= pr_threshold:
        return "⭐ Highly influential in this area"
    if sem_score > 0.75:
        return "🎯 Closely matches your query"
    query_cats  = set(df.iloc[query_idx]["categories"]) if query_idx is not None else set()
    shared_cats = query_cats & set(rec_row["categories"])
    if shared_cats:
        return f"📂 Trending in {list(shared_cats)[0]}"
    return "🔍 Related to your research area"


def recommend(
    query_vec,
    query_idx    = None,
    top_n        = 10,
    fetch_k      = 50,
    alpha        = 0.70,
    beta         = 0.20,
    lambda_mmr   = 0.60,
    recency_boost= 0.15,
    half_life    = 3.0,
    min_year     = 2000,
    max_year     = 2026,
    categories   = None,
):
    query_vec_f      = query_vec.reshape(1, -1).astype(np.float32)
    sem_scores, idxs = faiss_index.search(query_vec_f, fetch_k + 1)

    rows = []
    for i, s in zip(idxs[0], sem_scores[0]):
        idx = int(i)
        if idx == query_idx:
            continue
        row  = df.iloc[idx]
        year = pd.to_datetime(row["date"], errors="coerce")
        if pd.isna(year) or not (min_year <= year.year <= max_year):
            continue
        if categories and not set(row["categories"]) & set(categories):
            continue
        graph_s = float(row["graph_score"])
        rows.append({
            "cand_idx"       : idx,
            "paper_id"       : row["paper_id"],
            "title"          : row["title"],
            "categories"     : row["categories"],
            "date"           : row["date"],
            "graph_pagerank" : row["graph_pagerank"],
            "relevance_score": round(alpha * float(s) + beta * graph_s, 4),
        })

    if not rows:
        return pd.DataFrame()

    candidates    = pd.DataFrame(rows)
    cand_indices  = candidates["cand_idx"].tolist()
    mmr_results   = maximal_marginal_relevance(
        query_vec, cand_indices, embeddings,
        top_n=top_n * 2, lambda_param=lambda_mmr
    )
    mmr_idx_order = [r[0] for r in mmr_results]
    idx_to_row    = {r["cand_idx"]: r for _, r in candidates.iterrows()}
    reranked      = pd.DataFrame(
        [idx_to_row[i] for i in mmr_idx_order if i in idx_to_row]
    ).head(top_n * 2).reset_index(drop=True)

    boosted  = apply_recency_boost(reranked, half_life, recency_boost)
    boosted  = boosted.head(top_n).reset_index(drop=True)

    why_tags = []
    for _, row in boosted.iterrows():
        match = df[df["paper_id"] == row["paper_id"]].index
        if match.empty:
            why_tags.append("🔍 Related to your research area")
            continue
        rec_idx = int(match[0])
        why_tags.append(generate_explanation(query_idx, rec_idx, df, G, row["relevance_score"]))
    boosted["why_recommended"] = why_tags

    return boosted[[
        "paper_id", "title", "categories", "date",
        "relevance_score", "recency", "final_score", "why_recommended"
    ]]


if "read_indices"   not in st.session_state: st.session_state.read_indices   = []
if "pinned_indices" not in st.session_state: st.session_state.pinned_indices = []
if "last_results"   not in st.session_state: st.session_state.last_results   = None

def get_profile_vector():
    all_indices = st.session_state.read_indices.copy()
    weights     = [1.0] * len(all_indices)
    for idx in st.session_state.pinned_indices:
        if idx in all_indices:
            weights[all_indices.index(idx)] = 3.0
        else:
            all_indices.append(idx)
            weights.append(3.0)
    if not all_indices:
        return None
    vecs    = embeddings[all_indices]
    weights = np.array(weights) / sum(weights)
    profile = np.average(vecs, axis=0, weights=weights)
    return (profile / np.linalg.norm(profile)).astype(np.float32)

with st.sidebar:
    st.title("⚙️ Settings")
    st.markdown("---")

    st.subheader("🎛️ Pipeline Weights")
    alpha      = st.slider("Semantic weight (α)",  0.0, 1.0, config["alpha"],      0.05)
    beta       = st.slider("Graph weight (β)",      0.0, 1.0, config["beta"],       0.05)
    lambda_mmr = st.slider("MMR diversity (λ)",     0.0, 1.0, config["lambda_mmr"], 0.05)
    recency_b  = st.slider("Recency boost",         0.0, 0.5, config["recency_boost"], 0.05)
    top_n      = st.slider("Results to show",       5,   20,  config["top_n"])

    st.markdown("---")
    st.subheader("🔍 Filters")
    year_range    = st.slider("Publication year", 2000, 2024, (2000, 2024))
    selected_cats = st.multiselect(
        "Filter by category",
        options      = list(ML_CATEGORIES.keys()),
        format_func  = lambda x: ML_CATEGORIES[x],
        default      = [],
    )

    st.markdown("---")
    st.subheader("👤 Your Profile")
    st.metric("Papers read",   len(st.session_state.read_indices))
    st.metric("Papers pinned", len(st.session_state.pinned_indices))
    if st.button("🗑️ Clear profile", use_container_width=True):
        st.session_state.read_indices   = []
        st.session_state.pinned_indices = []
        st.session_state.last_results   = None
        st.rerun()

st.title("📄 Research Paper Recommender")
st.caption("Semantic search · Citation graph · MMR diversity · Recency boost")
st.markdown("---")

tab1, tab2, tab3 = st.tabs(["🔎 Search", "👤 My Profile", "📊 System Stats"])

with tab1:
    col1, col2 = st.columns([3, 1])
    with col1:
        query_text = st.text_area(
            "Paste a title, abstract, or research question:",
            height      = 120,
            placeholder = "e.g. attention mechanisms in transformer models...",
        )
    with col2:
        st.markdown("<br>", unsafe_allow_html=True)
        n_read      = len(st.session_state.read_indices)
        use_profile = st.checkbox(
            "Blend with my profile",
            value    = n_read > 0,
            disabled = n_read == 0,
        )
        search_btn = st.button("🔍 Search", use_container_width=True, type="primary")

    if search_btn and query_text.strip():
        with st.spinner("Finding papers..."):
            query_vec = model.encode(
                [query_text], normalize_embeddings=True, convert_to_numpy=True
            ).astype(np.float32)[0]

            if use_profile and n_read > 0:
                profile_vec = get_profile_vector()
                if profile_vec is not None:
                    query_vec = 0.6 * query_vec + 0.4 * profile_vec
                    query_vec = query_vec / np.linalg.norm(query_vec)

            results = recommend(
                query_vec    = query_vec,
                top_n        = top_n,
                alpha        = alpha,
                beta         = beta,
                lambda_mmr   = lambda_mmr,
                recency_boost= recency_b,
                min_year     = year_range[0],
                max_year     = year_range[1],
                categories   = selected_cats if selected_cats else None,
            )
            st.session_state.last_results = results

    if st.session_state.last_results is not None:
        results = st.session_state.last_results
        if results.empty:
            st.warning("No results — try adjusting filters.")
        else:
            st.markdown(f"### Top {len(results)} Recommendations")
            for rank, (_, row) in enumerate(results.iterrows(), 1):
                paper_id = row["paper_id"]
                match    = df[df["paper_id"] == paper_id].index
                df_idx   = int(match[0]) if not match.empty else None

                with st.container():
                    c1, c2 = st.columns([6, 1])
                    with c1:
                        st.markdown(f"**{rank}. {row['title']}**")
                        cats     = " · ".join(ML_CATEGORIES.get(c, c) for c in row["categories"])
                        date_str = str(row["date"])[:10]
                        st.caption(f"📂 {cats}   🗓️ {date_str}   {row['why_recommended']}")
                        ca, cb, cc = st.columns(3)
                        ca.metric("Relevance", f"{row['relevance_score']:.3f}")
                        cb.metric("Recency",   f"{row['recency']:.3f}")
                        cc.metric("Final",     f"{row['final_score']:.3f}")
                    with c2:
                        st.markdown("<br>", unsafe_allow_html=True)
                        if df_idx is not None:
                            if st.button("✅ Read", key=f"read_{rank}_{paper_id}", use_container_width=True):
                                if df_idx not in st.session_state.read_indices:
                                    st.session_state.read_indices.append(df_idx)
                                st.toast("Marked as read!")
                            if st.button("📌 Pin", key=f"pin_{rank}_{paper_id}", use_container_width=True):
                                if df_idx not in st.session_state.pinned_indices:
                                    st.session_state.pinned_indices.append(df_idx)
                                st.toast("Pinned!")
                        st.link_button("🔗 ArXiv", f"https://arxiv.org/abs/{paper_id}", use_container_width=True)
                    st.markdown("---")

with tab2:
    st.subheader("📚 Reading History")
    if not st.session_state.read_indices:
        st.info("No papers read yet. Search and click ✅ Read.")
    else:
        for df_idx in st.session_state.read_indices:
            row = df.iloc[df_idx]
            c1, c2 = st.columns([6, 1])
            with c1:
                pinned_tag = "📌 " if df_idx in st.session_state.pinned_indices else ""
                st.markdown(f"**{pinned_tag}{row['title']}**")
                st.caption(str(row["date"])[:10])
            with c2:
                if st.button("More like this", key=f"mlt_{df_idx}", use_container_width=True):
                    with st.spinner("Finding similar papers..."):
                        results = recommend(
                            query_vec = embeddings[df_idx],
                            query_idx = df_idx,
                            top_n     = top_n,
                            alpha     = alpha,
                            beta      = beta,
                            lambda_mmr= lambda_mmr,
                        )
                        st.session_state.last_results = results
                    st.rerun()
            st.markdown("---")

    st.subheader("📌 Pinned Papers")
    if not st.session_state.pinned_indices:
        st.info("No papers pinned yet.")
    else:
        for df_idx in st.session_state.pinned_indices:
            row = df.iloc[df_idx]
            st.markdown(f"📌 **{row['title']}**")
            st.caption(" · ".join(ML_CATEGORIES.get(c, c) for c in row["categories"]))
            st.markdown("---")

with tab3:
    st.subheader("📊 Dataset Overview")
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Total Papers",  f"{len(df):,}")
    c2.metric("FAISS Vectors", f"{faiss_index.ntotal:,}")
    c3.metric("Graph Nodes",   f"{G.number_of_nodes():,}")
    c4.metric("Graph Edges",   f"{G.number_of_edges():,}")

    st.markdown("---")
    st.subheader("📂 Papers per Category")
    cat_counts = {}
    for cats in df["categories"]:
        for c in cats:
            if c in ML_CATEGORIES:
                cat_counts[ML_CATEGORIES[c]] = cat_counts.get(ML_CATEGORIES[c], 0) + 1
    cat_df = pd.DataFrame(cat_counts.items(), columns=["Category", "Count"]).sort_values("Count", ascending=False)
    st.bar_chart(cat_df.set_index("Category"))

    st.markdown("---")
    st.subheader("⚙️ Active Config")
    st.json({
        "alpha"         : alpha,
        "beta"          : beta,
        "lambda_mmr"    : lambda_mmr,
        "recency_boost" : recency_b,
        "top_n"         : top_n,
        "year_range"    : year_range,
        "categories"    : selected_cats or "All",
    })

    st.markdown("---")
    st.subheader("📈 Model Comparison")
    try:
        with open(f"{ARTIFACTS_DIR}all_metrics.json") as f:
            all_metrics = json.load(f)
        rows = []
        for name, m in all_metrics.items():
            if "mean_precision@k" in m:
                rows.append({
                    "Model"        : name,
                    "Precision@10" : m["mean_precision@k"],
                    "Std"          : m.get("std", "-"),
                })
        if rows:
            st.dataframe(pd.DataFrame(rows), use_container_width=True)
    except FileNotFoundError:
        st.info("Metrics file not found.")

Writing /kaggle/working/app.py


In [7]:
from pyngrok import ngrok
import subprocess, time

ngrok.kill()

proc = subprocess.Popen(
    ["streamlit", "run", "/kaggle/working/app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.enableCORS", "false",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

time.sleep(5)
public_url = ngrok.connect(8501)
print(f"\nApp live at: {public_url}")
print("Keep this cell running.")


App live at: NgrokTunnel: "https://goal-gondola-geologic.ngrok-free.dev" -> "http://localhost:8501"
Keep this cell running.
